# Integration accuracy and order

A PROTOTYPE of the compute gate (Prompt 268, COMPUTE_GATE_II.md): it is not in the catalogue and not shipped. It reads its input as the text `gravitas_input` and its settings as the JSON text `gravitas_params`, and returns `gravitas_output`, a dict of numbers. Python only, numpy and scipy only.


In [ ]:
import json, math
import numpy as np
from scipy.integrate import solve_ivp
doc = json.loads(gravitas_input)
assert doc.get("format") == "gravitas.orbital-system" and doc.get("version") == 1, "this notebook reads gravitas.orbital-system/1"
init = doc.get("initial")
assert isinstance(init, dict), "the file has no initial state: save it again from the Orbital System Builder"
G = float(init["G"])
bodies = init["bodies"]
m = np.array([b["mass"] for b in bodies], float)
X0 = np.array([[b["x"], b["y"]] for b in bodies], float)
V0 = np.array([[b["vx"], b["vy"]] for b in bodies], float)
N = len(m)
assert 2 <= N <= 12, "from two to twelve bodies"
d0 = np.linalg.norm(X0[:, None, :] - X0[None, :, :], axis=2)
iu = np.triu_indices(N, 1)
assert d0[iu].min() > 1e-9 * d0[iu].max(), "refused: two bodies start at the same place"
k = int(np.argmin(d0[iu])); ia, ja = int(iu[0][k]), int(iu[1][k])
Ts = 2 * math.pi * math.sqrt(d0[ia, ja] ** 3 / (G * (m[ia] + m[ja])))
com = (m[:, None] * X0).sum(axis=0) / m.sum()
r_far = float(np.max(np.linalg.norm(X0 - com, axis=1)))

def accel(x):
    d = x[None, :, :] - x[:, None, :]
    r2 = (d ** 2).sum(axis=2)
    np.fill_diagonal(r2, 1.0)
    w = r2 ** -1.5
    np.fill_diagonal(w, 0.0)
    return G * ((m[None, :] * w)[:, :, None] * d).sum(axis=1)

def rhs(t, y):
    return np.concatenate([y[2 * N:], accel(y[:2 * N].reshape(N, 2)).ravel()])

ref = solve_ivp(rhs, (0, Ts), np.concatenate([X0.ravel(), V0.ravel()]), method="DOP853", rtol=1e-13, atol=1e-13)
Xref = ref.y[:2 * N, -1].reshape(N, 2)
def final_error(x):
    return float(np.max(np.linalg.norm(x - Xref, axis=1)))

In [ ]:
def euler(x, v, h):
    return x + h * v, v + h * accel(x)

def symplectic(x, v, h):
    v = v + h * accel(x)
    return x + h * v, v

def verlet(x, v, h):
    v = v + 0.5 * h * accel(x)
    x = x + h * v
    return x, v + 0.5 * h * accel(x)

def rk4(x, v, h):
    k1x, k1v = v, accel(x)
    k2x, k2v = v + 0.5 * h * k1v, accel(x + 0.5 * h * k1x)
    k3x, k3v = v + 0.5 * h * k2v, accel(x + 0.5 * h * k2x)
    k4x, k4v = v + h * k3v, accel(x + h * k3x)
    return (x + h / 6 * (k1x + 2 * k2x + 2 * k3x + k4x), v + h / 6 * (k1v + 2 * k2v + 2 * k3v + k4v))

STEPS = [50, 100, 200, 400, 800]
errors, order = {}, {}
for name, f in [("euler", euler), ("symplectic", symplectic), ("verlet", verlet), ("rk4", rk4)]:
    es = []
    for n in STEPS:
        h = Ts / n
        x, v = X0.copy(), V0.copy()
        for _ in range(n):
            x, v = f(x, v, h)
        es.append(final_error(x))
    errors[name] = es
    order[name] = float(np.polyfit(np.log([Ts / n for n in STEPS]), np.log(es), 1)[0])
gravitas_output = {"answer": {"Ts": Ts, "r_far": r_far, "steps": STEPS, "errors": errors, "order": order}}